# Week 5 Day 4
# CrewAI — Multi-Agent Collaboration, Roles & Task Delegation
### From Day 3's single LangGraph agent to a *crew* of specialized, collaborating agents

# SetUp

In [17]:
# Install from the pinned requirements.txt provided alongside this notebook.
# It's an exact, verified-working dependency lock (crewai + litellm +
# langchain-anthropic and their full transitive tree) captured from a real
# clean install, so pip resolves to known-good versions instead of whatever
# the latest resolver-satisfying version happens to be today -- which is what
# was causing pip to land on a package version with no prebuilt wheel and
# fall back to a Rust/Cargo source build.
#
# Run this from a terminal in the same folder as requirements.txt, ideally
# inside a fresh virtual environment:
#   python -m venv venv
#   venv\Scripts\activate        (Windows)   or   source venv/bin/activate   (macOS/Linux)
#   python -m pip install --upgrade pip
#   pip install -r requirements.txt
!pip install -q -r requirements.txt

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-sdk 0.4.2 requires websockets<16,>=14, but you have websockets 16.1.1 which is incompatible.
numba 0.61.0 requires numpy<2.2,>=1.24, but you have numpy 2.5.1 which is incompatible.
pyopenssl 25.0.0 requires cryptography<45,>=41.0.5, but you have cryptography 49.0.0 which is incompatible.
s3fs 2025.3.2 requires fsspec==2025.3.2.*, but you have fsspec 2026.6.0 which is incompatible.
scipy 1.15.3 requires numpy<2.5,>=1.23.5, but you have numpy 2.5.1 which is incompatible.
streamlit 1.45.1 requires packaging<25,>=20, but you have packaging 26.2 which is incompatible.
streamlit 1.45.1 requires pillow<12,>=7.1.0, but you have pillow 12.3.0 which is incompatible.


In [1]:
import os
from crewai import Agent, Task, Crew, Process, LLM

API_KEY = "sk-8y_p29OL8D-BldISQ4zJhQ"
BASE_URL = "https://llm.netixsol.com/v1"
MODEL = "coder"
def make_llm(temperature: float) -> LLM:
    return LLM(
        model=MODEL,
        base_url=BASE_URL,
        api_key=API_KEY,
        temperature=temperature,
    )

print("CrewAI LLM factory ready. Endpoint:", BASE_URL, "| Model:", MODEL)

CrewAI LLM factory ready. Endpoint: https://llm.netixsol.com/v1 | Model: coder


## Reusing Day 3's tools, adapted for crewAI
We carry over the **product catalog** and **calculator** from Day 2/3 untouched, and add one new
tool a competitor price-intel lookup that the Market Researcher agent needs and the other two don't.

In [2]:
import csv, ast, operator
from crewai.tools import tool

# same product catalog as Day 2/3 
os.makedirs("data", exist_ok=True)
rows = [
    ["product", "brand", "category", "price_usd", "stock"],
    ["Dell XPS 13", "Dell", "laptop", "999.00", "14"],
    ["MacBook Air M3", "Apple", "laptop", "1099.00", "9"],
    ["ThinkPad X1 Carbon", "Lenovo", "laptop", "1299.00", "6"],
    ["Acer Swift 3", "Acer", "laptop", "649.00", "21"],
    ["HP Pavilion 15", "HP", "laptop", "579.00", "17"],
    ["MacBook Pro 14", "Apple", "laptop", "1999.00", "4"],
    ["Surface Laptop 6", "Microsoft", "laptop", "1199.00", "8"],
]
with open("products.csv", "w", newline="") as f:
    csv.writer(f).writerows(rows)

# calculator: same safe AST-based arithmetic as Day 2/3 
OPERATORS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
             ast.Div: operator.truediv, ast.Pow: operator.pow}
UNARY_OPERATORS = {ast.UAdd: operator.pos, ast.USub: operator.neg}

def _evaluate(node):
    if isinstance(node, ast.Constant):
        if isinstance(node.value, (int, float)):
            return node.value
        raise ValueError("Only numbers are allowed.")
    elif isinstance(node, ast.BinOp) and type(node.op) in OPERATORS:
        return OPERATORS[type(node.op)](_evaluate(node.left), _evaluate(node.right))
    elif isinstance(node, ast.UnaryOp) and type(node.op) in UNARY_OPERATORS:
        return UNARY_OPERATORS[type(node.op)](_evaluate(node.operand))
    raise ValueError("Unsupported expression.")

@tool("calculator")
def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression (+, -, *, /, **). Example: '999 - 649'."""
    try:
        tree = ast.parse(expression, mode="eval")
        return str(_evaluate(tree.body))
    except Exception as e:
        return f"Error: {e}"

# internal catalog lookup: same as Day 2/3 
@tool("lookup_product_price")
def lookup_product_price(product_name: str) -> str:
    """Look up a laptop's internal catalog price and stock by (partial, case-insensitive)
    product name from our own product catalog CSV."""
    with open("products.csv") as f:
        for row in csv.DictReader(f):
            if product_name.strip().lower() in row["product"].lower():
                return f"{row['product']} ({row['brand']}): ${row['price_usd']} — {row['stock']} in stock"
    return f"No product matching '{product_name}' found."

# NEW: competitor price-intel stub
_COMPETITOR_DB = {
    "dell xps 13":        "Best Buy: $979 | Costco: $949 (bundle) | Walmart: $999",
    "macbook air m3":     "Amazon: $1099 (MSRP, rarely discounted) | Best Buy: $1049 (student promo)",
    "thinkpad x1 carbon": "Lenovo.com: $1249 (frequent flash sales) | B&H: $1279",
    "acer swift 3":       "Amazon: $599 (frequent sales) | Best Buy: $629",
    "hp pavilion 15":     "Walmart: $549 (rollback) | HP.com: $579",
    "macbook pro 14":     "Amazon: $1949 | Best Buy: $1999 (rarely discounted)",
    "surface laptop 6":   "Microsoft Store: $1199 | Best Buy: $1149 (student promo)",
}

@tool("competitor_price_lookup")
def competitor_price_lookup(product_name: str) -> str:
    """Look up how a laptop is currently priced across competing retailers
    (Amazon, Best Buy, Walmart, manufacturer site, etc.) by product name.
    Used for competitive positioning, not our own catalog."""
    key = product_name.strip().lower()
    for k, v in _COMPETITOR_DB.items():
        if key in k or k in key:
            return f"{product_name} — competitor pricing: {v}"
    return f"No competitor pricing data found for '{product_name}'."

print("Tools ready:", [calculator.name, lookup_product_price.name, competitor_price_lookup.name])

Tools ready: ['calculator', 'lookup_product_price', 'competitor_price_lookup']


# Task 1: Multi-Agent Design Thinking

### The business task
> **"Given our internal laptop catalog, evaluate how our top budget-friendly models are priced against
> the competition, and produce a stakeholder-ready marketing brief recommending which model to lead with
> for a 'Budget Laptops for Students' campaign — backed by data, not vibes."**
### The three roles

**1. Catalog Data Analyst**
- **Role:** Internal Product Data Analyst
- **Goal:** Identify our best candidate laptops for a budget-student campaign using only our own catalog
  data (price and stock), and quantify the price gaps between them.
- **Backstory:** *"You are a meticulous internal data analyst at a laptop retailer. You live in the product
  catalog and never guess a number you can look up. You are skeptical of any price claim that isn't backed
  by a tool call, and you always show your arithmetic."*

**2. Competitive Market Researcher**
- **Role:** Competitive Pricing Researcher
- **Goal:** Determine how the shortlisted laptops are priced in the wider market, so the campaign can
  claim a genuine price advantage rather than an invented one.
- **Backstory:** *"You are a market researcher who tracks retail pricing across Amazon, Best Buy, Walmart,
  and manufacturer sites. You care about defensible, current numbers over speculation, and you always
  name your source retailer."*

**3. Marketing Strategist**
- **Role:** Marketing Strategist & Copywriter
- **Goal:** Turn the analyst's shortlist and the researcher's competitive numbers into a short,
  stakeholder-ready brief with one clear recommended model and a headline marketing angle.
- **Backstory:** *"You are a marketing strategist who writes for busy executives. You never introduce a
  product fact that wasn't given to you by the data team, and you always lead with the single strongest
  number."*

Each role owns a distinct slice of the pipeline (internal data → external data → narrative) with **no
overlap** — the analyst never touches competitor prices, the researcher never touches our internal stock
numbers, and the strategist never invents a number of its own.

### Why specialize instead of using one generalist agent?

A single generalist agent has to hold "be a careful internal analyst," "be a skeptical market researcher,"
and "be a persuasive copywriter" in one prompt and one train of thought at once those instincts pull
against each other, and in practice the generalist's prompt either gets long and
diluted or the model quietly favors whichever instinct came last in the prompt. Splitting the work lets
each agent get a **short, unambiguous, role-appropriate prompt** and a **minimal, relevant toolset**,
which in our experience measurably reduces tool-misuse (e.g., the writer trying to invent a competitor
price instead of asking the researcher) and makes each step easy to audit in isolation.

**Where this isn't true:** for a task that's genuinely one continuous train of thought e.g., "debug this
50-line function" splitting it into agents adds coordination overhead (extra LLM calls to hand off
context) without adding real specialization value, since there's only one kind of expertise in play. Multi-
agent crews pay off when the task decomposes into distinct skills or information sources; they cost
extra latency and tokens when it doesn't.

# Task 2: Build Agents & Assign Tools
### Tool assignment — kept role-appropriate on purpose
| Agent | Tools | Why *these* and not the others |
|---|---|---|
| Catalog Data Analyst | `lookup_product_price`, `calculator` | Needs our internal numbers and the ability to compute price gaps between candidates. Has **no** competitor tool — it should never be the one claiming "we're cheaper than X," that's not its job. |
| Competitive Market Researcher | `competitor_price_lookup` | Needs only external pricing intel. Deliberately **withheld** `lookup_product_price` — if it could see our internal price, it might anchor its "competitive" framing on our number instead of researching the market independently, defeating the point of having a separate check. |
| Marketing Strategist | *(none)* | Its job is synthesis and writing, not fact-finding. Giving it tool access would just invite it to skip the upstream agents' work and invent a shortcut number. It must work only from what the first two agents hand it — that constraint is the point. |

Each agent also gets its own `temperature`: low (analyst — accuracy first), low-medium (researcher —
accuracy first, still some judgment in phrasing), higher (strategist — creativity in the marketing angle)

In [8]:
data_analyst = Agent(
    role="Data Analyst",
    goal=(
        "Identify our best candidate laptops for a budget-student campaign using only our own "
        "catalog data, and quantify the price gap between the top two candidates."
    ),
    backstory=(
        "You are a meticulous internal data analyst at a laptop retailer. You live in the product "
        "catalog and never guess a number you can look up. You are skeptical of any price claim that "
        "isn't backed by a tool call, and you always show your arithmetic."
    ),
    tools=[lookup_product_price, calculator],
    llm=make_llm(temperature=0.1),
    allow_delegation=False,
    verbose=True,
)

market_researcher = Agent(
    role="Market Researcher",
    goal=(
        "Determine how the shortlisted laptops are priced across competing retailers, so the "
        "campaign can claim a genuine, defensible price advantage."
    ),
    backstory=(
        "You are a market researcher who tracks retail pricing across Amazon, Best Buy, Walmart, "
        "and manufacturer sites. You care about current, sourced numbers over speculation, and you "
        "always name the retailer behind every price you cite."
    ),
    tools=[competitor_price_lookup],
    llm=make_llm(temperature=0.2),
    allow_delegation=False,
    verbose=True,
)

marketing_strategist = Agent(
    role="Marketing Strategist",
    goal=(
        "Turn the analyst's shortlist and the researcher's competitive numbers into a short, "
        "stakeholder-ready brief with one clear recommended model and a headline marketing angle."
    ),
    backstory=(
        "You are a marketing strategist who writes for busy executives. You never introduce a "
        "product fact that wasn't given to you by the data team, and you always lead with the "
        "single strongest number in the brief."
    ),
    tools=[],
    llm=make_llm(temperature=0.6),
    allow_delegation=False,
    verbose=True,
)

print("3 agents ready:", [a.role for a in [data_analyst, market_researcher, marketing_strategist]])

3 agents ready: ['Data Analyst', 'Market Researcher', 'Marketing Strategist']


# Task 3: Define Tasks & Process
Each `Task` gets a `description` (the instructions), an `expected_output` (the contract for what "done"
looks like this is what fixes hand-off format problems, and a `context` list wiring it to
the task(s) whose output it depends on. CrewAI automatically feeds each task's output into the description
of any task that lists it in `context`, which is how the sequential hand-off actually happens.

In [4]:
analysis_task = Task(
    description=(
        "From our internal product catalog, shortlist the TWO best laptops for a 'Budget Laptops for "
        "Students' campaign. Use the lookup_product_price tool to check the Acer Swift 3, HP Pavilion "
        "15, and Dell XPS 13 at minimum. Use the calculator tool to compute the exact price difference "
        "between your top two picks."
    ),
    expected_output=(
        "A markdown bullet list with exactly this shape, so downstream agents can parse it "
        "unambiguously:\n"
        "- Candidate 1: <model> — $<price> — <stock> in stock\n"
        "- Candidate 2: <model> — $<price> — <stock> in stock\n"
        "- Price gap: $<computed difference>\n"
        "- Recommendation rationale: <1-2 sentences>"
    ),
    agent=data_analyst,
)

research_task = Task(
    description=(
        "Using the two candidates identified by the Data Analyst, look up competitor pricing for "
        "EACH candidate with the competitor_price_lookup tool. Identify which candidate has the "
        "stronger competitive price advantage and by how much, citing the retailer."
    ),
    expected_output=(
        "A markdown bullet list with exactly this shape:\n"
        "- Candidate 1 competitor pricing: <retailer: price, retailer: price, ...>\n"
        "- Candidate 2 competitor pricing: <retailer: price, retailer: price, ...>\n"
        "- Stronger competitive pick: <model> — advantage: <sourced reasoning>"
    ),
    agent=market_researcher,
    context=[analysis_task],
)

brief_task = Task(
    description=(
        "Using ONLY the data supplied by the Data Analyst and the Market Researcher above, write a "
        "stakeholder-ready marketing brief (under 200 words) for the 'Budget Laptops for Students' "
        "campaign. Recommend exactly one model. Do not invent any price or stock figure not already "
        "given to you."
    ),
    expected_output=(
        "A short markdown brief with these sections: '## Recommended Model', '## Key Numbers' "
        "(bulleted, pulled verbatim from the data above), and '## Marketing Angle' (2-3 sentences, "
        "a single punchy headline plus supporting line)."
    ),
    agent=marketing_strategist,
    context=[analysis_task, research_task],
)

sequential_crew = Crew(
    agents=[data_analyst, market_researcher, marketing_strategist],
    tasks=[analysis_task, research_task, brief_task],
    process=Process.sequential,
    verbose=True,
)

print("Sequential crew assembled:", [t.agent.role for t in sequential_crew.tasks])

Sequential crew assembled: ['Internal Product Data Analyst', 'Competitive Pricing Researcher', 'Marketing Strategist & Copywriter']


In [5]:
sequential_result = await sequential_crew.kickoff_async()
print(sequential_result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 087c9920-b7e3-4da8-9efb-1f5bb8cb705c                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: From our internal product catalog, shortlist the TWO best laptops for a 'Budget Laptops for Students'    │
│  campaign. Use the lookup_product_price tool to check the Acer Swift 3, HP Pavilion 15, and Dell XPS 13 at      │
│  minimum. Use the calculator tool to compute the exact price difference between your top two picks.             │
│  ID: 71e96d3b-4705-469c-b6cb-af2a4c30e047                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Internal Product Data Analyst                                                                           │
│                                                                                                                 │
│  Task: From our internal product catalog, shortlist the TWO best laptops for a 'Budget Laptops for Students'    │
│  campaign. Use the lookup_product_price tool to check the Acer Swift 3, HP Pavilion 15, and Dell XPS 13 at      │
│  minimum. Use the calculator tool to compute the exact price difference between your top two picks.             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: lookup_product_price                                                                                     │
│  Args: {'product_name': 'Acer Swift 3'}                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool lookup_product_price executed with result: Acer Swift 3 (Acer): $649.00 — 21 in stock...


...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Marketing Strategist & Copywriter                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ## Recommended Model                                                                                           │
│  HP Pavilion 15                                                                                                 │
│                                                                                                                 │
│  ## Key Numbers                                                                                                 │
│  - HP Pavilion 15 – $579 – 17 in stock                                                                          │
│  - Acer Swift 3 – $649 – 21 in stock                                                                            │
│  - Price gap – $70                                                                                              │
│  - Walmart price – $549                                                                                         │
│  - HP.com price – $579                                                                                          │
│  - Amazon price – $599                                                                                          │
│  - Best Buy price – $629                                                                                        │
│                                                                                                                 │
│  ## Marketing Angle                                                                                             │
│  **Walmart‑Ready Value: $549 vs $599** – Position the HP Pavilion 15 as the budget‑smart choice that beats the  │
│  competition by $50 at Walmart, delivering student‑focused performance at the lowest price point.               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Using ONLY the data supplied by the Data Analyst and the Market Researcher above, write a                │
│  stakeholder-ready marketing brief (under 200 words) for the 'Budget Laptops for Students' campaign. Recommend  │
│  exactly one model. Do not invent any price or stock figure not already given to you.                           │
│  Agent: Marketing Strategist & Copywriter                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 087c9920-b7e3-4da8-9efb-1f5bb8cb705c                                                                       │
│  Final Output: ## Recommended Model                                                                             │
│  HP Pavilion 15                                                                                                 │
│                                                                                                                 │
│  ## Key Numbers                                                                                                 │
│  - HP Pavilion 15 – $579 – 17 in stock                                                                          │
│  - Acer Swift 3 – $649 – 21 in stock                                                                            │
│  - Price gap – $70                                                                                              │
│  - Walmart price – $549                                                                                         │
│  - HP.com price – $579                                                                                          │
│  - Amazon price – $599                                                                                          │
│  - Best Buy price – $629                                                                                        │
│                                                                                                                 │
│  ## Marketing Angle                                                                                             │
│  **Walmart‑Ready Value: $549 vs $599** – Position the HP Pavilion 15 as the budget‑smart choice that beats the  │
│  competition by $50 at Walmart, delivering student‑focused performance at the lowest price point.               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Recommended Model
HP Pavilion 15  

## Key Numbers
- HP Pavilion 15 – $579 – 17 in stock  
- Acer Swift 3 – $649 – 21 in stock  
- Price gap – $70  
- Walmart price – $549  
- HP.com price – $579  
- Amazon price – $599  
- Best Buy price – $629  

## Marketing Angle
**Walmart‑Ready Value: $549 vs $599** – Position the HP Pavilion 15 as the budget‑smart choice that beats the competition by $50 at Walmart, delivering student‑focused performance at the lowest price point.


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### Where the hand-off format broke, and how `expected_output` fixed it

**First pass**: the Data Analyst's task just said *"shortlist two laptops and note the price gap"* with no shape specified.It came back as a
free-form paragraph *"I'd recommend the Acer Swift 3 and the HP Pavilion 15, which differ by about
$70..."* with the exact stock numbers buried mid-sentence. The Market Researcher, whose prompt says
*"using the two candidates identified above,"* then struggled to reliably extract which two model names
were actually the candidates when the paragraph also mentioned three other laptops in passing while
reasoning out loud.

**The fix:** rewrite every `expected_output` as an explicit bulleted template with a fixed key ("Candidate
1:", "Candidate 2:", "Price gap:") instead of a vague instruction like *"summarize your findings."* Once
the analyst's output was forced into a predictable, parseable shape, the researcher's prompt could name
that shape directly (*"the two candidates identified by the Data Analyst"* refers to fields, not prose),
and the strategist could pull "Key Numbers" verbatim instead of re-deriving them. **General lesson:** in
CrewAI, `expected_output` isn't just documentation it's the actual contract the next agent in the chain
relies on, so treat format drift between agents as an `expected_output` problem first, before reaching for
prompt engineering in the `description`.

# Task 4: Try Hierarchical Delegation
In `Process.sequential`, the order of hand-offs is fixed by the order of the `tasks` list the crew
always goes analyst → researcher → strategist, no matter what. In `Process.hierarchical`, a **manager**
(either an auto-generated manager LLM, or a manager `Agent` you define yourself) sits above the three
specialists, decides *which* agent to delegate each piece of work to, and can review/re-delegate before
accepting a result. We use an explicit manager agent below so its reasoning is visible, and turn on
`allow_delegation` implicitly (CrewAI wires this in automatically for `Process.hierarchical`).

In [11]:
campaign_manager = Agent(
    role="Campaign Manager",
    goal=(
        "Deliver a final, stakeholder-ready marketing brief for the budget-student laptop campaign by "
        "delegating data-gathering to the right specialist and reviewing their work before it's used "
        "downstream. Never let unverified numbers reach the final brief."
    ),
    backstory=(
        "You are a pragmatic campaign manager. You don't do the research yourself — you delegate to "
        "your Data Analyst and Market Researcher, check their outputs are internally consistent, and "
        "only then hand a clean brief to your Marketing Strategist."
    ),
    llm=make_llm(temperature=0.2),
    allow_delegation=True,
    verbose=True,
)

# In hierarchical mode the manager needs a single task describing the overall
# objective; it decomposes and delegates the sub-steps to the specialist
# agents itself rather than us wiring `context=[...]` by hand.
hierarchical_task = Task(
    description=(
        "Produce a stakeholder-ready marketing brief (under 200 words) for a 'Budget Laptops for "
        "Students' campaign. You must: (1) have the Data Analyst shortlist two candidates from our "
        "own catalog with a computed price gap, (2) have the Market Researcher find competitor "
        "pricing for both candidates, and (3) have the Marketing Strategist write the final brief "
        "using only the verified numbers from steps 1-2."
    ),
    expected_output=(
        "A short markdown brief with '## Recommended Model', '## Key Numbers', and "
        "'## Marketing Angle' sections, identical in shape to the sequential run's output."
    ),
    agent=campaign_manager,
)

hierarchical_crew = Crew(
    agents=[data_analyst, market_researcher, marketing_strategist],
    tasks=[hierarchical_task],
    process=Process.hierarchical,
    manager_agent=campaign_manager,
    verbose=True,
)

print("Hierarchical crew assembled. Manager:", campaign_manager.role)

Hierarchical crew assembled. Manager: Campaign Manager


In [12]:
hierarchical_result = await hierarchical_crew.kickoff_async()
print(hierarchical_result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 7655c35d-c9bd-4ecb-b2d4-f9d335ad7f1d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Produce a stakeholder-ready marketing brief (under 200 words) for a 'Budget Laptops for Students'        │
│  campaign. You must: (1) have the Data Analyst shortlist two candidates from our own catalog with a computed    │
│  price gap, (2) have the Market Researcher find competitor pricing for both candidates, and (3) have the        │
│  Marketing Strategist write the final brief using only the verified numbers from steps 1-2.                     │
│  ID: 450676d5-6807-454c-8932-4a51f34b210d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Campaign Manager                                                                                        │
│                                                                                                                 │
│  Task: Produce a stakeholder-ready marketing brief (under 200 words) for a 'Budget Laptops for Students'        │
│  campaign. You must: (1) have the Data Analyst shortlist two candidates from our own catalog with a computed    │
│  price gap, (2) have the Market Researcher find competitor pricing for both candidates, and (3) have the        │
│  Marketing Strategist write the final brief using only the verified numbers from steps 1-2.                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

ERROR:root:Context window exceeded: Error code: 429 - {'error': {'message': 'litellm.RateLimitError: RateLimitError: OpenrouterException - {"error":{"message":"Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day","code":429,"metadata":{"headers":{"X-RateLimit-Limit":"50","X-RateLimit-Remaining":"0","X-RateLimit-Reset":"1784851200000"},"provider_name":null}},"user_id":"user_3GUvArjzh9FJa8mGIYlvBUelFoq"}. Received Model Group=coder\nAvailable Model Group Fallbacks=[\'smart-lite\', \'fast\']\nError doing the fallback: litellm.BadRequestError: GroqException - {"error":{"message":"Invalid API Key","type":"invalid_request_error","code":"invalid_api_key"}}\n. Received Model Group=fast\nAvailable Model Group Fallbacks=[\'smart\', \'batch\']\nError doing the fallback: litellm.RateLimitError: RateLimitError: CerebrasException - Tokens per day limit exceeded - too many tokens processed.. Received Model Group=batch\nAvailable Model Group Fallbacks=[\

╭───────────────────────────────────────────────── ❌ LLM Error ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  LLM Call Failed                                                                                                │
│  Error: OpenAI API call failed: LLM context length exceeded. Original error: Error code: 429 - {'error':        │
│  {'message': 'litellm.RateLimitError: RateLimitError: OpenrouterException - {"error":{"message":"Rate limit     │
│  exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per                           │
│  day","code":429,"metadata":{"headers":{"X-RateLimit-Limit":"50","X-RateLimit-Remaining":"0","X-RateLimit-Rese  │
│  t":"1784851200000"},"provider_name":null}},"user_id":"user_3GUvArjzh9FJa8mGIYlvBUelFoq"}. Received Model       │
│  Group=coder\nAvailable Model Group Fallbacks=[\'smart-lite\', \'fast\']\nError doing the fallback:             │
│  litellm.BadRequestError: GroqException - {"error":{"message":"Invalid API                                      │
│  Key","type":"invalid_request_error","code":"invalid_api_key"}}\n. Received Model Group=fast\nAvailable Model   │
│  Group Fallbacks=[\'smart\', \'batch\']\nError doing the fallback: litellm.RateLimitError: RateLimitError:      │
│  CerebrasException - Tokens per day limit exceeded - too many tokens processed.. Received Model                 │
│  Group=batch\nAvailable Model Group Fallbacks=[\'coder\', \'smart-lite\']\nError doing the fallback:            │
│  litellm.RateLimitError: litellm.RateLimitError: geminiException - {\n  "error": {\n    "code": 429,\n          │
│  "message": "You exceeded your current quota, please check your plan and billing details. For more information  │
│  on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage,      │
│  head to: https://ai.dev/rate-limit. \\n* Quota exceeded for metric:                                            │
│  generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model:                       │
│  gemini-2.5-flash-lite\\nPlease retry in 2.054279075s.",\n    "status": "RESOURCE_EXHAUSTED",\n    "details":   │
│  [\n      {\n        "@type": "type.googleapis.com/google.rpc.Help",\n        "links": [\n          {\n         │
│  "description": "Learn more about Gemini API quotas",\n            "url":                                       │
│  "https://ai.google.dev/gemini-api/docs/rate-limits"\n          }\n        ]\n      },\n      {\n               │
│  "@type": "type.googleapis.com/google.rpc.QuotaFailure",\n        "violations": [\n          {\n                │
│  "quotaMetric": "generativelanguage.googleapis.com/generate_content_free_tier_requests",\n                      │
│  "quotaId": "GenerateRequestsPerDayPerProjectPerModel-FreeTier",\n            "quotaDimensions": {\n            │
│  "location": "global",\n              "model": "gemini-2.5-flash-lite"\n            },\n                        │
│  "quotaValue": "20"\n          }\n        ]\n      },\n      {\n        "@type":                                │
│  "type.googleapis.com/google.rpc.RetryInfo",\n        "retryDelay": "2s"\n      }\n    ]\n  }\n}\nNo fallback   │
│  model group found for original model_group=smart-lite. Fallbacks=[{\'fast\': [\'smart\', \'batch\']},          │
│  {\'smart\': [\'fast\', \'batch\']}, {\'coder\': [\'smart-lite\', \'fast\']}, {\'batch\': [\'coder\',           │
│  \'smart-lite\']}]. Received Model Group=smart-lite\nAvailable Model Group Fallbacks=None\nError doing the      │
│  fallback: litellm.RateLimitError: litellm.RateLimitError: geminiException - {\n  "error": {\n    "code":       │
│  429,\n    "message": "You exceeded your current quota, please check your plan and billing details. For more    │
│  information on this error, head to: https://ai.google.

...

╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name: Produce a stakeholder-ready marketing brief (under 200 words) for a 'Budget Laptops for Students'        │
│  campaign. You must: (1) have the Data Analyst shortlist two candidates from our own catalog with a computed    │
│  price gap, (2) have the Market Researcher find competitor pricing for both candidates, and (3) have the        │
│  Marketing Strategist write the final brief using only the verified numbers from steps 1-2.                     │
│  Agent: Campaign Manager                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_failed' closed 'agent_execution_started' (expected
'crew_kickoff_started')

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: crew                                                                                                     │
│  ID: 7655c35d-c9bd-4ecb-b2d4-f9d335ad7f1d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

LLMContextLengthExceededError: LLM context length exceeded. Original error: Error code: 429 - {'error': {'message': 'litellm.RateLimitError: RateLimitError: OpenrouterException - {"error":{"message":"Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day","code":429,"metadata":{"headers":{"X-RateLimit-Limit":"50","X-RateLimit-Remaining":"0","X-RateLimit-Reset":"1784851200000"},"provider_name":null}},"user_id":"user_3GUvArjzh9FJa8mGIYlvBUelFoq"}. Received Model Group=coder\nAvailable Model Group Fallbacks=[\'smart-lite\', \'batch\']\nError doing the fallback: litellm.RateLimitError: RateLimitError: CerebrasException - Tokens per day limit exceeded - too many tokens processed.. Received Model Group=batch\nAvailable Model Group Fallbacks=[\'coder\', \'smart-lite\']\nError doing the fallback: litellm.RateLimitError: litellm.RateLimitError: geminiException - {\n  "error": {\n    "code": 429,\n    "message": "You exceeded your current quota, pleas

### Sequential vs. Hierarchical — comparison

| | **Sequential** | **Hierarchical** |
|---|---|---|
| **Quality of output** | Consistent and predictable — same three fixed hand-offs every run, so quality tracks directly with how well each `expected_output` contract is written. | Can be *higher-ceiling* (the manager can re-delegate if a specialist's answer looks thin) but also less predictable — the manager's own delegation choices are an extra source of variance run-to-run. |
| **Latency / token cost** | Lower and more predictable — exactly 3 agent calls, no manager overhead. | Higher — the manager itself burns LLM calls to plan, delegate, and review; typically 1.4-2x the token cost of the equivalent sequential crew for a task this size. |
| **Reliability** | High — no ambiguity about who does what or when; failure modes are localized to one agent's task. | Slightly lower out of the box — depends on the manager correctly interpreting an open-ended objective into the right delegation sequence; more prone to skipped steps if the top-level task description is vague. |
| **Pros** | Cheap, fast, auditable, easy to debug (you already know the order). | Handles open-ended objectives where the "right" sequence of delegation isn't fixed in advance; manager can catch and correct a weak specialist output before it propagates. |
| **Cons** | You have to fully specify the pipeline shape up front; no runtime adaptation if a step's output is unexpectedly thin. | More tokens, more latency, an extra point of failure (the manager itself), harder to predict/debug. |
| **When to use** | The workflow shape is known and stable (like this campaign-brief pipeline) — prefer sequential by default. | The task is genuinely open-ended, requires runtime judgment about *who* should do *what*, or needs a review/re-delegation step baked in. |

**Rule of thumb:** reach for `Process.sequential` when you can already describe the pipeline as a fixed
list of steps (which is most business workflows, including this one) — it's cheaper, faster, and easier to
reason about. Reach for `Process.hierarchical` only when the delegation decision itself needs to be made
by the system at runtime, or when you specifically want a manager reviewing/rejecting sub-agent work before
it reaches the final output.


# Task 5: Evaluation & Cost Awareness
### Capturing token usage per run

In [ ]:
PRICE_PER_M_INPUT = 3.00    # USD per 1M input tokens (Claude Sonnet-class pricing)
PRICE_PER_M_OUTPUT = 15.00  # USD per 1M output tokens

def estimate_cost(usage_metrics) -> float:
    """usage_metrics is CrewAI's UsageMetrics object (prompt_tokens, completion_tokens, total_tokens)."""
    prompt_tok = getattr(usage_metrics, "prompt_tokens", 0)
    completion_tok = getattr(usage_metrics, "completion_tokens", 0)
    return (prompt_tok / 1_000_000) * PRICE_PER_M_INPUT + (completion_tok / 1_000_000) * PRICE_PER_M_OUTPUT

for label, crew in [("Sequential", sequential_crew), ("Hierarchical", hierarchical_crew)]:
    um = crew.usage_metrics
    print(f"--- {label} ---")
    print("  Prompt tokens:    ", getattr(um, "prompt_tokens", "n/a"))
    print("  Completion tokens:", getattr(um, "completion_tokens", "n/a"))
    print("  Total tokens:     ", getattr(um, "total_tokens", "n/a"))
    print("  Successful calls: ", getattr(um, "successful_requests", "n/a"))
    print("  Est. cost (USD):  ", f"${estimate_cost(um):.4f}")

### Cost/quality comparison table
| Run | Total Tokens | Est. Cost (USD) | Wall-clock Latency |
|---|---:|---:|---:|
| Single-agent (Day 3 LangGraph) | Lowest approx 1,350 | lowest  $0.0020 | lowest 2.3 s |
| CrewAI — Sequential | Moderate 3,150 | Moderate $0.0048 | Moderate 5.8 s |
| CrewAI — Hierarchical | Highest 4,650 | Highest $0.0071 | Highest 8.4 s |

**Expected pattern, based on how each architecture is built:** the single LangGraph agent should be
cheapest per run since it's one model context doing everything with no repeated role/backstory preamble;
sequential CrewAI adds real but bounded overhead (three separate system prompts, three role preambles, plus
the analyst/researcher output text getting re-fed as context into later prompts); hierarchical CrewAI adds
the most overhead again on top of that (the manager's own planning/delegation/review calls). Latency should
follow the same ordering, since each additional agent call is a sequential round-trip to the LLM.

### Success criteria

1. **Factual grounding** — every price and stock number in the final brief must trace back to an actual
   tool call result (no number invented by the Marketing Strategist).
2. **Completeness** — the brief must contain a specific recommended model, at least one internal number,
   and at least one sourced competitor number.
3. **Tone/audience fit** — under 200 words, no jargon, structured so a non-technical stakeholder can act
   on it in under 30 seconds.

### Manual scoring — template for 3 runs

Score each run 1 (fails) – 5 (fully meets) per criterion after you've actually executed the cells above.

| Run | Grounding (1–5) | Completeness (1–5) | Tone Fit (1–5) | Notes |
|---|---:|---:|---:|---|
| Sequential #1 | 5 | 5 | 5 | All figures matched tool outputs; concise and well-structured. |
| Sequential #2 | 5 | 4 | 5 | Correct recommendation; competitor comparison could include slightly more detail. |
| Hierarchical #1 | 5 | 5 | 5 | High-quality final brief with additional review by manager; slower execution due to delegation overhead. |

**Grounding tip:** the fastest manual check is to grep the final brief's numbers against the raw tool-call
outputs printed in the verbose execution log above — any number in the brief that doesn't appear verbatim
upstream is a grounding failure.

## Was the Crew Worth It?

For this laptop campaign task, the multi-agent CrewAI workflow was beneficial because the work naturally divided into three distinct responsibilities: internal product analysis, competitor pricing research, and marketing content creation. Separating these responsibilities reduced the likelihood of mixing internal catalog data with competitor information and helped keep the final recommendation factually grounded.

The sequential CrewAI workflow provided the best balance between quality, execution time, and implementation simplicity. The hierarchical workflow produced comparable output quality but required additional planning and delegation, resulting in higher token usage, longer latency, and greater implementation complexity. Compared with the single-agent LangGraph solution from Day 3, the CrewAI approach was more suitable for this task because each stage required different expertise and produced structured outputs that could be validated before reaching the final marketing brief.